                # LabeledFewShot

                Sample labeled training examples and attach them directly as demonstrations; no teacher calls are needed during compilation.

                **Use it when:** You have trustworthy labels, want the cheapest few-shot baseline, and do not need generated reasoning traces.

                **What compilation changes:** Demonstrations only; the original instruction remains unchanged.

                Important configuration in this benchmark:

                - `k=4` caps prompt growth
- `sample=True` samples from the frozen training split

                Every notebook loads the same 74-row dataset and frozen, pair-grouped
                train/validation/test membership before it can compile anything.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'labeled-few-shot'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='labeled-few-shot'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


                ## Compile shape

                This is the essential DSPy call used by the shared executable runner:

                ```python
                optimizer = dspy.LabeledFewShot(k=4)
optimized_detector = optimizer.compile(detector, trainset=trainset, sample=True)
                ```

                `compile` returns a program. The shared runner then evaluates that program on the
                untouched 20-example test split. The baseline has its own notebook; all other
                notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: labeled-few-shot
task model: openai/gpt-5.6-luna
final test accuracy: 75.0% (15/20)
optimization time: 0.0s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/labeled-few-shot.json
- canonical prompt: chapter06/results/final_prompts/labeled-few-shot.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Inspect the four saved demos. This method can improve in-context calibration, but sampled examples are not selected against the validation set.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 4
1. is_ai=False: join is downstream of follow_branch_a and branch_false. The join task will show up as skipped because its trigger_rule is set to all_success by default, and the skip caused by t…
2. is_ai=False: A given blog article might have various statuses - for instance, it might be visible to everyone (i.e. public), or only visible to the author (i.e. private). It may also be hidd…
3. is_ai=False: You can adjust the configuration of cmake variables optionally (without building first), by doing the following. For example, adjusting the pre-detected directories for CuDNN or…
4. is_ai=False: When we submit the form, the POST /articles request is mapped to the create action. The create action does attempt to save @article. Therefore, validations are checked. If any v…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.